In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 日本語フォント設定 (フォールバック付き)
try:
    plt.rcParams["font.family"] = "IPAGothic"
except Exception:
    plt.rcParams["font.family"] = ["Noto Sans CJK JP", "DejaVu Sans"]
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.grid.alpha"] = 0.3
plt.rcParams["figure.dpi"] = 120

In [ ]:
from sqlalchemy import text

from db.connection import DatabaseConnection


def get_engine():
    """PostgreSQL接続エンジンを取得 (settings.yaml 使用)"""
    conn = DatabaseConnection()
    return conn.get_engine()


def load_races(engine, start: str, end: str) -> pd.DataFrame:
    """レースデータをロード"""
    return pd.read_sql(text("""
        SELECT * FROM raw.races
        WHERE race_date BETWEEN :start AND :end
        ORDER BY race_date
    """), engine, params={"start": start, "end": end})


def load_entries(engine, race_ids: list[str]) -> pd.DataFrame:
    """出走馬データをロード"""
    return pd.read_sql(text("""
        SELECT * FROM raw.entries
        WHERE race_id = ANY(:race_ids)
    """), engine, params={"race_ids": race_ids})


print("Setup complete. PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
def generate_mock_race_df(n_races: int = 100) -> pd.DataFrame:
    """テスト用の合成レースデータを生成 (DB不要)"""
    np.random.seed(42)
    rows = []
    for i in range(n_races):
        n = np.random.randint(10, 18)
        for j in range(n):
            rows.append({
                "race_id": f"R{i:04d}",
                "race_date": pd.Timestamp("2020-01-01") + pd.Timedelta(days=i),
                "umaban": j + 1,
                "finish_pos": j + 1,
                "win_odds": max(1.1, np.random.lognormal(2.0, 0.8)),
                "popularity_rank": j + 1,
                "surface": np.random.choice(["turf", "dirt"]),
                "distance_bin": np.random.choice(["sprint", "mile", "intermediate", "long"]),
                "field_size": n,
            })
    return pd.DataFrame(rows)


print("Mock data generator ready.")